Importing packages

In [8]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
user_path = "/Users/piotr/"

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
def embed_column(df: pd.DataFrame, col: str, model: SentenceTransformer) -> pd.DataFrame:
    texts = df[col].fillna("").astype(str).tolist()
    emb = model.encode(texts, show_progress_bar=True)
    df[f"{col}_emb"] = list(emb)
    return df

In [11]:
def explode_embeddings(df: pd.DataFrame, emb_cols: list[str]) -> pd.DataFrame:
    for col in emb_cols:
        if col not in df.columns:
            continue
            
        non_null = df[col].dropna()
        if non_null.empty:
            continue
            
        first_valid = non_null.iloc[0]
        emb_dim = len(first_valid)

        # Use apply instead of fillna
        def _ensure_array(x):
            if isinstance(x, (np.ndarray, list)) and x is not None:
                return x
            return np.zeros(emb_dim, dtype=float)
        
        emb_series = df[col].apply(_ensure_array)
        emb_matrix = np.vstack(emb_series.values)

        for i in range(emb_dim):
            df[f"{col}_{i}"] = emb_matrix[:, i]

        df.drop(columns=[col], inplace=True)

    return df

In [12]:
def fill_na_comprehensive(
    df: pd.DataFrame,
    embedding_cols: list[str] | None = None,
    string_fill: str = "",
    bool_fill: int | bool = 0,
    num_fill: float | int = 0.0,
) -> pd.DataFrame:

    df = df.copy()

    if embedding_cols is not None:
        for col in embedding_cols:
            if col not in df.columns:
                continue

            non_null = df[col].dropna()
            if non_null.empty:
                continue

            first_valid = non_null.iloc[0]
            emb_dim = len(first_valid)

            def _fill_emb(x):
                if isinstance(x, (np.ndarray, list)):
                    return x
                return np.zeros(emb_dim, dtype=float)

            df[col] = df[col].apply(_fill_emb)

    bool_cols = [c for c in df.columns if df[c].dtype == "bool"]

    bin_int_cols = []
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c].dtype):
            uniq = pd.Series(df[c].dropna().unique())
            if len(uniq) <= 2 and set(uniq.tolist()).issubset({0, 1}):
                bin_int_cols.append(c)

    df[bool_cols] = df[bool_cols].fillna(bool_fill)
    df[bin_int_cols] = df[bin_int_cols].fillna(bool_fill)

    obj_cols = [c for c in df.columns if df[c].dtype == "object" or str(df[c].dtype) == "string"]
    if embedding_cols is not None:
        obj_cols = [c for c in obj_cols if c not in embedding_cols]

    df[obj_cols] = df[obj_cols].fillna(string_fill)

    num_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c].dtype)
        and c not in bin_int_cols
    ]
    df[num_cols] = df[num_cols].fillna(num_fill)

    return df

In [13]:
def data_fe(df: pd.DataFrame, model) -> pd.DataFrame:
    df = df.copy()

    df = df[df['volume_total'] > 0]

    df["snapshot_ts"] = pd.to_datetime(df["snapshot_ts"], errors="coerce")

    df["year"] = df["snapshot_ts"].dt.year
    df["month"] = df["snapshot_ts"].dt.month
    df["day"] = df["snapshot_ts"].dt.day
    df["hour"] = df["snapshot_ts"].dt.hour
    df["minute"] = df["snapshot_ts"].dt.minute
    df["second"] = df["snapshot_ts"].dt.second

    if "Unnamed: 0" in df.columns:
        df.drop(columns="Unnamed: 0", inplace=True)

    all_null_cols = df.columns[df.isna().all()]
    df.drop(columns=all_null_cols, inplace=True)

    threshold = 0.85
    null_ratio = df.isna().mean()
    cols_to_drop = null_ratio[null_ratio > threshold].index
    df.drop(columns=cols_to_drop, inplace=True)

    df = df.sort_values(["market_id", "snapshot_ts"])

    start = (
        df[df["active"]]
        .groupby("market_id")["snapshot_ts"]
        .min()
        .rename("start_time")
    )

    end = (
        df[df["closed"]]
        .groupby("market_id")["snapshot_ts"]
        .min()
        .rename("end_time")
    )

    result = (
        pd.concat([start, end], axis=1)
        .dropna(subset=["start_time", "end_time"])
    )

    result["time_active_hours"] = (
        result["end_time"] - result["start_time"]
    ) / pd.Timedelta(hours=1)

    df["time_active"] = df["market_id"].map(result["time_active_hours"])

    text_cols = [
        "event_slug",
        "event_title",
        "market_slug",
        "question",
        "resolution_source",
        "event_description",
        "promotion_group_y",
    ]

    for col in text_cols:
        if col in df.columns:
            df = embed_column(df, col, model)

    date_cols = ["snapshot_ts"]
    for col in date_cols:
        if col in df.columns:
            df[col] = df[col].astype("int64") // 10**9

    bool_cols = df.select_dtypes(include=["bool"]).columns
    for col in bool_cols:
        df[col] = df[col].astype("uint8")

    emb_cols = [f"{c}_emb" for c in text_cols if f"{c}_emb" in df.columns]

    df = fill_na_comprehensive(
        df,
        embedding_cols=emb_cols,
        string_fill="",
        bool_fill=0,
        num_fill=0.0,
    )

    df = explode_embeddings(df, emb_cols)

    for col in list(df.columns):
        if df[col].dtype == "object" or str(df[col].dtype) == "string":
            if col == "market_id":
                df[col] = df[col].astype("int64")
            else:
                df.drop(columns=col, inplace=True)

    return df


In [ ]:
db_additional = pd.read_csv(f"{user_path}UEP-MachineLearning/data_model/db_additional.csv")
db_collected = pd.read_csv(f"{user_path}UEP-MachineLearning/data_model/db_collected.csv")

/var/folders/dk/zjwq8trx6cbbykf77jz61hnc0000gn/T/ipykernel_72373/2261972198.py:2: DtypeWarning: Columns (0: restricted, 1: promotion_group_x, 2: event_description, 3: resolved_by, 4: promotion_group_y) have mixed types. Specify dtype option on import or set low_memory=False.
  db_collected = pd.read_csv("/Users/piotr/UEP-MachineLearning/data_model/db_collected.csv")


In [15]:
db_additional = data_fe(db_additional, model=model)
db_collected = data_fe(db_collected, model=model)

Batches: 100%|██████████| 2670/2670 [00:27<00:00, 97.42it/s] 
/var/folders/dk/zjwq8trx6cbbykf77jz61hnc0000gn/T/ipykernel_72373/2987519751.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_{i}"] = emb_matrix[:, i]


In [ ]:
db_additional.to_csv(f"{user_path}UEP-MachineLearning/data_model/db_additional.csv")
db_collected.to_csv(f"{user_path}UEP-MachineLearning/data_model/db_collected.csv")